# Disaster tweet triage
**AlertMedia take-home · Sr AI/ML Engineer · Ryder Dale Walton**

The notebook follows the exercise:

| Section | Exercise step |
| --- | --- |
| §1 Data | (setup for everything else) |
| §2 Evaluation | Step 1: metrics and the cost of errors |
| §3 Models | Step 2: baseline and one improvement |
| §4 Error analysis | Step 3 |

Steps 4 (LLM evaluation harness) and 5 (two-week plan) are in the README.
Run top to bottom. The fine-tuning cells use Apple Silicon (MPS) or CUDA when available and save
their predictions to `results/`, so later sections can run without retraining.

In [ ]:
# §0 Setup
import html, io, re, zipfile
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

HF_REPO = "mhamza-007/binary-class-tweets-dataset"
HF_FILE = "disastrous_tweets.zip"

# How to resolve texts that appear with both labels (see §1):
#   "majority": keep the majority label, drop exact ties    "drop": drop every conflicting text
CONFLICT_POLICY = "majority"

pd.set_option("display.max_colwidth", 110)

## §1 Data

The Hub repo stores the data as a single zip, `disastrous_tweets.zip`, so `datasets.load_dataset`
fails on it (and the Hub's dataset viewer is broken). We download the zip and read the CSV directly.
It is a byte-for-byte copy of Kaggle's *NLP with Disaster Tweets*. Its `test.csv` has no labels,
so all evaluation below uses splits of `train.csv`.

In [ ]:
def load_zip() -> zipfile.ZipFile:
    local = DATA_DIR / HF_FILE
    if not local.exists():
        from huggingface_hub import hf_hub_download
        local = Path(hf_hub_download(HF_REPO, HF_FILE, repo_type="dataset", local_dir=DATA_DIR))
    return zipfile.ZipFile(local)

with load_zip() as z:
    print({i.filename: i.file_size for i in z.infolist()})
    raw = pd.read_csv(io.BytesIO(z.read("train.csv")))

print(f"{len(raw):,} labelled tweets · {raw.target.mean():.1%} positive (target = 1)")
raw.head()

In [ ]:
# location is free text and about a third missing, so the models ignore it
raw.isna().sum().to_frame("missing").T

**Every tweet was collected by searching for a disaster keyword.** So the real task is telling a
literal use of a disaster word ("forest fire near La Ronge") from a figurative one ("this mixtape
is fire"), not finding disasters in a general feed. Some keywords nearly decide the label alone,
which is a shortcut that would not survive a broader feed.

In [ ]:
kw = raw.groupby("keyword").target.agg(positive_rate="mean", tweets="size").sort_values("positive_rate")
pd.concat([kw.head(5), kw.tail(5)]).round(2)

### Normalize and deduplicate

Retweets and copy-pasted headlines often differ only in their URLs, @handles or HTML entities.
Left in, the same tweet can land in both train and test and inflate every score. We normalize,
collapse each group of duplicates to one row, and resolve texts that carry both labels with
`CONFLICT_POLICY`.

In [ ]:
def normalize(text: str) -> str:
    """Lowercase, decode HTML entities, mask URLs and @handles, collapse whitespace."""
    text = html.unescape(text).lower()
    text = re.sub(r"https?://\S+", " httpurl ", text)
    text = re.sub(r"@\w+", " @user ", text)
    return re.sub(r"\s+", " ", text).strip()

df = raw.assign(norm=raw.text.map(normalize))
groups = df.groupby("norm").target.agg(copies="size", positive_rate="mean")
conflicts = groups[(groups.positive_rate > 0) & (groups.positive_rate < 1)]

print(f"{len(df):,} rows → {len(groups):,} unique normalized texts ({len(df) - len(groups):,} duplicate copies)")
print(f"{len(conflicts)} texts appear with both labels ({conflicts.copies.sum()} rows); "
      f"{(conflicts.positive_rate == 0.5).sum()} are exact ties")
conflicts.sort_values("copies", ascending=False).head(5).round(2)

In [ ]:
def dedupe(df: pd.DataFrame, policy: str = CONFLICT_POLICY) -> pd.DataFrame:
    """One row per normalized text; label conflicts resolved by `policy`."""
    g = df.groupby("norm").agg(text=("text", "first"), keyword=("keyword", "first"),
                               copies=("target", "size"), positive_rate=("target", "mean"))
    if policy == "majority":
        g = g[g.positive_rate != 0.5]            # a tie can't be resolved by vote
    elif policy == "drop":
        g = g[g.positive_rate.isin([0, 1])]      # keep unanimous texts only
    else:
        raise ValueError(policy)
    return (g.assign(target=(g.positive_rate > 0.5).astype(int),
                     label_from_vote=~g.positive_rate.isin([0, 1]))
             .drop(columns="positive_rate").reset_index())

data = dedupe(df)
print(f"{CONFLICT_POLICY!r}: {len(data):,} rows kept · {data.target.mean():.1%} positive · "
      f"{data.label_from_vote.sum()} labels set by majority vote")

### Split

70 / 15 / 15, stratified by label and **grouped by a coarser near-duplicate key** (punctuation,
"RT" prefixes and URL/handle tokens stripped), so near-copies never straddle train and test.
Thresholds and model choices are made on validation. Test is scored once per model, at the end.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def near_dup_key(norm_text: str) -> str:
    t = re.sub(r"^rt\b", "", norm_text)
    t = re.sub(r"[^a-z0-9 ]", " ", t)
    t = re.sub(r"\b(httpurl|user)\b", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def grouped_split(data: pd.DataFrame, seed: int = SEED) -> np.ndarray:
    """≈70/15/15 split from 20 stratified, grouped folds: 3 test, 3 validation, 14 train."""
    folds = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=seed)
    fold = np.empty(len(data), dtype=int)
    for k, (_, idx) in enumerate(folds.split(data, data.target, data.group)):
        fold[idx] = k
    return np.select([fold < 3, fold < 6], ["test", "val"], "train")

data["group"] = data.norm.map(near_dup_key)
data["split"] = grouped_split(data)
assert data.groupby("group").split.nunique().max() == 1, "a near-duplicate group crosses splits"

train, val, test = (data[data.split == s].reset_index(drop=True) for s in ["train", "val", "test"])
data.groupby("split").target.agg(rows="size", positive_rate="mean").loc[["train", "val", "test"]].round(3)

**Prevalence caveat.** About 41% of these tweets are disasters only because every one was
pre-selected by a disaster keyword. In a live feed the share is far lower, so precision measured
here overstates production precision. §2 shows the correction.

## §2 Evaluation (Step 1)

**The model is a triage layer, not the sender.** Analysts review threats before customers are
alerted, so the classifier decides what analysts *see*:

- A **false positive** costs analyst minutes here (and alert fatigue if it ever reached customers).
- A **false negative** is a threat to people that no analyst ever sees. It can't be recovered later.

So the goal is **maximum recall while the analyst queue stays useful**: precision is a capacity
constraint, not the objective. The README explains each choice.

| Role | Metric |
| --- | --- |
| Headline, threshold-free | Average precision (PR-AUC), with ROC-AUC alongside |
| Operating point | Recall at precision ≥ 0.80, threshold chosen on validation |
| Single-number summary | F2 (recall weighted 2×), plus F1 for comparison with published work |
| Business | Expected cost per 1,000 tweets, a miss = 10, a false alarm = 1 |
| Trust in scores | Brier score and a reliability curve |
| Uncertainty | Bootstrap 95% intervals; paired tests when comparing models (§3) |

In [ ]:
# Evaluation settings (my choices; see README, Step 1)
PRECISION_FLOOR = 0.80            # at least 80% of the analyst queue should be real disasters
COST_FN, COST_FP = 10, 1          # a missed disaster costs 10x a false alarm an analyst dismisses
PREVALENCES = [0.01, 0.05, 0.10]  # plausible real-world disaster rates for the precision correction
N_BOOT = 1000

In [ ]:
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,
                             fbeta_score, precision_recall_curve, roc_auc_score)

def pick_threshold(y, p, floor=PRECISION_FLOOR) -> float:
    """The threshold with the highest recall whose precision is still >= floor. Use on validation only."""
    prec, rec, thr = precision_recall_curve(y, p)
    ok = prec[:-1] >= floor
    if not ok.any():
        raise ValueError(f"no threshold reaches precision {floor}")
    return float(thr[np.argmax(np.where(ok, rec[:-1], -1))])

def evaluate(y, p, threshold) -> dict:
    y, p = np.asarray(y), np.asarray(p)
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "avg_precision": average_precision_score(y, p),
        "roc_auc": roc_auc_score(y, p),
        "precision": tp / max(tp + fp, 1),
        "recall": tp / max(tp + fn, 1),
        "f1": fbeta_score(y, pred, beta=1, zero_division=0),
        "f2": fbeta_score(y, pred, beta=2, zero_division=0),
        "brier": brier_score_loss(y, p),
        "false_positive_rate": fp / max(fp + tn, 1),
        "cost_per_1k": 1000 * (COST_FN * fn + COST_FP * fp) / len(y),
        "alerts_per_1k": 1000 * (tp + fp) / len(y),
    }

def bootstrap(y, p, threshold, n_boot=N_BOOT, seed=SEED) -> pd.DataFrame:
    """Point estimates with 95% percentile intervals, resampling tweets with replacement."""
    y, p = np.asarray(y), np.asarray(p)
    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        draws.append(evaluate(y[i], p[i], threshold))
    draws = pd.DataFrame(draws)
    return pd.DataFrame({"value": evaluate(y, p, threshold),
                         "ci_low": draws.quantile(0.025), "ci_high": draws.quantile(0.975)})

**Precision at real-world prevalence.** Holding a model's recall and false-positive rate fixed,
precision at prevalence π is `recall·π / (recall·π + FPR·(1−π))`. Section 3 applies this to each model.

In [ ]:
def precision_at_prevalence(recall, fpr, prevalences=PREVALENCES) -> dict:
    return {f"precision at {pi:.0%} prevalence": recall * pi / (recall * pi + fpr * (1 - pi))
            for pi in prevalences}

# Example: 80% recall with a 10% false-positive rate
pd.Series(precision_at_prevalence(0.80, 0.10)).round(3)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

def plot_pr_and_reliability(models: dict, y):
    """models: name -> predicted probabilities for the same tweets."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
    for name, p in models.items():
        prec, rec, _ = precision_recall_curve(y, p)
        ax1.plot(rec, prec, label=f"{name} (AP {average_precision_score(y, p):.3f})")
        observed, predicted = calibration_curve(y, p, n_bins=10, strategy="quantile")
        ax2.plot(predicted, observed, marker="o", ms=4, label=name)
    ax1.axhline(PRECISION_FLOOR, ls="--", c="grey", lw=1)
    ax1.text(0.02, PRECISION_FLOOR + 0.01, f"precision floor {PRECISION_FLOOR:.2f}", color="grey", fontsize=9)
    ax1.set(xlabel="Recall", ylabel="Precision", title="Precision–recall", xlim=(0, 1), ylim=(0.35, 1.01))
    ax2.plot([0, 1], [0, 1], ls="--", c="grey", lw=1)
    ax2.set(xlabel="Predicted probability", ylabel="Observed disaster rate", title="Reliability")
    ax1.legend(loc="lower left")
    ax2.legend(loc="upper left")
    fig.tight_layout()
    return fig

**Sanity check.** Before training anything, score a trivial model on validation: each tweet gets
its keyword's disaster rate in train. This tests the harness and measures how strong the keyword
shortcut is.

In [ ]:
kw_rate = train.groupby(train.keyword.fillna("none")).target.mean()
p_keyword = val.keyword.fillna("none").map(kw_rate).fillna(train.target.mean())
print(f"keyword-only, validation: average precision {average_precision_score(val.target, p_keyword):.3f}, "
      f"ROC-AUC {roc_auc_score(val.target, p_keyword):.3f}  "
      f"(chance: {val.target.mean():.3f} and 0.500)")

In [ ]:
RESULTS, TEST_PROBS = {}, {}

def score_model(name, p_val, p_test) -> pd.DataFrame:
    """Pick the operating threshold on validation, then score test once, with bootstrap intervals."""
    t = pick_threshold(val.target, p_val)
    RESULTS[name] = bootstrap(test.target, p_test, t)
    RESULTS[name].attrs["threshold"] = t
    TEST_PROBS[name] = np.asarray(p_test)
    print(f"{name}: threshold {t:.3f} (chosen on validation for precision ≥ {PRECISION_FLOOR})")
    return RESULTS[name].round(3)